In [ ]:
# Lab type: review
# Course: AI-Assisted Data Science
# Lesson: Evaluating AI-Generated Data Code — The Review Protocol
# Task: Apply the six-point review protocol to a flawed AI-generated preprocessing pipeline

# Lab: Reviewing AI-Generated Preprocessing Code

In this lab you'll apply the six-point review protocol to an AI-generated preprocessing pipeline.
The code is syntactically valid and will run without errors — your job is to find the **semantic**
errors that would silently corrupt a model's evaluation.

Work through each step in order. Do not skip ahead to the fix before completing the review.

## Setup: load the dataset

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/SophiArch/notebooks/main/datasets/employee_records.csv"
base = pd.read_csv(url)

# Add a binary target: promoted (1) when performance is 'exceeds_expectations', else 0
rng = np.random.default_rng(42)
base["promoted"] = (
    base["performance"].map(
        {"exceeds_expectations": 1, "meets_expectations": 0, "below_expectations": 0}
    )
    # Inject some noise so the target isn't perfectly deterministic
    .where(rng.random(len(base)) > 0.05, other=lambda s: 1 - s)
    .astype(int)
)

# Introduce realistic missing values
missing_idx = rng.choice(base.index, size=60, replace=False)
base.loc[missing_idx[:30], "salary"] = np.nan
base.loc[missing_idx[30:], "performance"] = np.nan

df = base.copy()
print(df.shape)
df.head()

## The code to review

A colleague used an AI tool to generate the preprocessing pipeline below.
Read it — **do not run it yet** — and work through the protocol in the steps that follow.

In [ ]:
# --- AI-GENERATED CODE: do not run until Step 6 ---
#
# from sklearn.preprocessing import StandardScaler, OrdinalEncoder
# from sklearn.impute import SimpleImputer
# from sklearn.model_selection import train_test_split
#
# # Separate target
# y = df["promoted"]
# X = df.drop("promoted", axis=1)
#
# # Impute numeric columns
# numeric_cols = X.select_dtypes(include="number").columns
# cat_cols = X.select_dtypes(include="object").columns
#
# imputer = SimpleImputer(strategy="mean")
# X[numeric_cols] = imputer.fit_transform(X[numeric_cols])
#
# # Encode categorical columns
# encoder = OrdinalEncoder()
# X[cat_cols] = encoder.fit_transform(X[cat_cols])
#
# # Scale numeric columns
# scaler = StandardScaler()
# X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
#
# # Split
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

print("Code read. Continue to Step 1.")

## Step 1: Read the code before you run it

Re-read the commented-out code above top to bottom.
List every decision the AI made that you didn't explicitly ask for.

*(Write your observations here — aim for at least three.)*

- 
- 
- 

## Step 2: Check for leakage

Identify the order of operations in the AI-generated pipeline:
when does the train/test split happen relative to `fit_transform`?

> **Question:** Does `fit_transform` run before or after `train_test_split`?
> Why does this matter for the model's evaluation metrics?

*(Write your answer here.)*

## Step 3: Verify column handling

Inspect the columns that will end up in `X` after the split.

In [ ]:
# Reconstruct X as the AI code would create it (before any transformations)
y = df["promoted"]
X = df.drop("promoted", axis=1)

print("Columns in X:", X.columns.tolist())
print("Target column:", y.name)
print("\nColumn dtypes:")
print(X.dtypes)

> **Question:** Are there any columns in `X` that should not be fed to a model?
> What would happen if `employee_id` were included as a feature?
> The categorical columns include `performance` — is ordinal encoding appropriate for all of them?

*(Write your answer here.)*

## Step 4: Inspect missing value handling

In [ ]:
# Missing values in X before any transformation
missing = X.isna().sum()
print("Missing values per column:")
print(missing[missing > 0])

> **Question:** The AI uses `SimpleImputer(strategy="mean")` only for numeric columns.
> What happens to missing values in categorical columns (like `performance`) under this pipeline?
> Is `strategy="mean"` always sensible for numeric columns in a dataset like this?

*(Write your answer here.)*

## Step 5: Spot-check sample output

Run a minimal version of the AI pipeline (still on the full dataset, as the AI wrote it)
and inspect the output.

In [ ]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer

X_check = X.copy()

numeric_cols = X_check.select_dtypes(include="number").columns
cat_cols = X_check.select_dtypes(include="object").columns

# Impute numeric columns (as the AI wrote)
imputer = SimpleImputer(strategy="mean")
X_check[numeric_cols] = imputer.fit_transform(X_check[numeric_cols])

# Encode categoricals (as the AI wrote)
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_check[cat_cols] = encoder.fit_transform(X_check[cat_cols])

# Scale numerics (as the AI wrote)
scaler = StandardScaler()
X_check[numeric_cols] = scaler.fit_transform(X_check[numeric_cols])

print(X_check.head())
print("\nDescribe:")
X_check.describe().round(3)

> **Question:** Does the `employee_id` column look like a useful feature after scaling?
> Check the mean and standard deviation of `employee_id` vs `salary` — what do you notice?
> Check the unique values in the encoded `performance` column — does the numeric ordering make sense?

In [ ]:
# Inspect the encoded performance column
print("Encoded performance categories (index → value):")
for i, cat in enumerate(encoder.categories_[list(cat_cols).index("performance")]):
    print(f"  {i}: {cat}")

> **Question:** `OrdinalEncoder` assigns numeric codes alphabetically by default.
> What order did it assign to the performance categories?
> Is that ordering meaningful for a model?

*(Write your answer here.)*

## Step 6: Validate a known case

Use domain knowledge to stress-test the output.
Employees with `performance == 'exceeds_expectations'` were most likely to be promoted.

In [ ]:
# Check: do high-performing employees have higher promotion rates in the original data?
print(df.groupby("performance")["promoted"].mean().sort_values(ascending=False))

In [ ]:
# Now confirm the encoded value for 'exceeds_expectations' is the highest ordinal
# (it should be if the encoding is meaningful — check whether it is)
perf_col_idx = list(cat_cols).index("performance")
cats = encoder.categories_[perf_col_idx]
print("OrdinalEncoder order for performance:", list(cats))
print("Encoded value for 'exceeds_expectations':",
      list(cats).index("exceeds_expectations"))

> **Question:** Is `exceeds_expectations` encoded with the highest numeric value?
> If not, a linear model would learn the wrong direction of the relationship.
> What fix would you apply?

*(Write your answer here.)*

## Fix: corrected pipeline

Rewrite the pipeline addressing all the issues you found:

1. **Leakage** — split before fitting any transformer
2. **Column exclusion** — drop `employee_id` before modelling
3. **Imputation of categoricals** — use `most_frequent` for the `performance` column
4. **Ordered encoding for `performance`** — supply the correct category order to `OrdinalEncoder`
5. **Nominal columns** — use one-hot encoding for `department`, `role`, and `region`

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

# --- Step 1: define features and target, drop identifier columns ---
y = df["promoted"]
X = df.drop(columns=["promoted", "employee_id"])  # drop target and identifier

# --- Step 2: split BEFORE any fitting ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Step 3: impute ---
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
ordinal_col = "performance"
nominal_cols = [c for c in X_train.select_dtypes(include="object").columns
                if c != ordinal_col]

num_imputer = SimpleImputer(strategy="median")   # median is more robust to salary outliers
X_train[numeric_cols] = num_imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = num_imputer.transform(X_test[numeric_cols])   # transform only

cat_imputer = SimpleImputer(strategy="most_frequent")
X_train[[ordinal_col]] = cat_imputer.fit_transform(X_train[[ordinal_col]])
X_test[[ordinal_col]]  = cat_imputer.transform(X_test[[ordinal_col]])

# --- Step 4: encode performance with the correct order ---
perf_order = [["below_expectations", "meets_expectations", "exceeds_expectations"]]
ord_enc = OrdinalEncoder(categories=perf_order)
X_train[[ordinal_col]] = ord_enc.fit_transform(X_train[[ordinal_col]])
X_test[[ordinal_col]]  = ord_enc.transform(X_test[[ordinal_col]])

# --- Step 5: one-hot encode nominal columns ---
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe_train = pd.DataFrame(
    ohe.fit_transform(X_train[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X_train.index
)
ohe_test = pd.DataFrame(
    ohe.transform(X_test[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X_test.index
)
X_train = pd.concat([X_train.drop(columns=nominal_cols), ohe_train], axis=1)
X_test  = pd.concat([X_test.drop(columns=nominal_cols),  ohe_test],  axis=1)

# --- Step 6: scale numeric columns (fit on train only) ---
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

print("Train shape:", X_train.shape)
print("Test shape: ", X_test.shape)
print("\nMissing values in train:", X_train.isna().sum().sum())
print("Missing values in test: ", X_test.isna().sum().sum())

## Verify the fix

Run the checks you'd apply in a real review.

In [ ]:
# Check 1: performance column is correctly ordered (0 = below, 1 = meets, 2 = exceeds)
sample = pd.DataFrame(
    {"performance": ["below_expectations", "meets_expectations", "exceeds_expectations"]}
)
print("Ordinal check:")
print(pd.concat([sample, pd.Series(ord_enc.transform(sample).ravel(), name="encoded")], axis=1))

In [ ]:
# Check 2: no train statistics leaked into test — scaler mean should be ~0 on train, not test
print("Train salary mean (should be ~0 after scaling): ",
      X_train["salary"].mean().round(4))
print("Test salary mean (will differ from 0 — that is correct):",
      X_test["salary"].mean().round(4))

In [ ]:
# Check 3: employee_id is absent from the feature matrix
assert "employee_id" not in X_train.columns, "employee_id should have been dropped"
assert "promoted" not in X_train.columns,    "promoted should not be in X"
print("Column checks passed.")
print("Final features:", X_train.columns.tolist())

## Reflection

Write 3–5 bullet points summarising what the original AI-generated code got wrong
and why each error matters in practice.

*(Write your reflection here.)*

- 
- 
- 